# Kelmarsh Wind Farm: 72-Hour Power Forecasting Pipeline

This notebook organizes the complete data pipeline and baseline modeling workflow for the **Kelmarsh Wind Farm** (6 Senvion MM92 turbines, 12.3 MW total rated capacity):
1. **Environment & Path Setup**
2. **Data Ingestion & Merging** (`eda.py`)
3. **Exploratory Data Analysis** (`analysis.py`)
4. **Data Preprocessing & Feature Engineering** (`preprocess.py`)
5. **Data Visualization** (`plot_turbine_comparison.py`, `plot_diurnal_seasonal_heatmap.py`)
6. **Chronological Splitting & Feature Scaling** (`prepare_scaling.py`)
7. **Baseline MLP Model Training & Evaluation** (`baseline_model.py`)
8. **Baseline Random Forest Model & Comparison with MLP**

## 1. Environment & Path Setup
Resolve project directories (`BASE_DIR`, `DATA_DIR`, `MODELS_DIR`) for Jupyter execution and import shared libraries.

In [ ]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Resolve project root whether notebook is run from root or src/
BASE_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "src" else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"BASE_DIR:   {BASE_DIR}")
print(f"DATA_DIR:   {DATA_DIR}")
print(f"MODELS_DIR: {MODELS_DIR}")

## 2. Data Ingestion & Merging (`eda.py`)
Read the 6 raw 10-minute Kelmarsh SCADA CSV files, select the core telemetry columns, label each turbine (`Turbine 1` to `Turbine 6`), save `data/merged_turbines_data.csv`, and print an initial summary.

In [ ]:
COLS = [
    "# Date and time",
    "Wind speed (m/s)",
    "Wind direction (°)",
    "Power (kW)",
    "Nacelle ambient temperature (°C)",
]

# 1. Read and merge all 6 turbine CSV files
files = sorted(glob.glob(os.path.join(DATA_DIR, "Turbine_Data_Kelmarsh_*_2024-*.csv")))
all_dfs = []

for i, f in enumerate(files, start=1):
    print(f"Reading Turbine {i}...")
    df_t = pd.read_csv(f, skiprows=9, usecols=COLS)
    df_t.columns = ["date_time", "wind_speed", "wind_direction", "power", "temperature"]
    df_t = df_t.dropna(subset=["date_time", "wind_speed"]).copy()
    df_t["turbine"] = f"Turbine {i}"
    all_dfs.append(df_t)

merged_df = pd.concat(all_dfs, ignore_index=True)

# 2. Save merged dataset
out_file = os.path.join(DATA_DIR, "merged_turbines_data.csv")
merged_df.to_csv(out_file, index=False)
print(f"\nSaved merged data to: {out_file} ({len(merged_df):,} rows)")

# 3. Simple EDA summary
print("\n--- DATASET SUMMARY ---")
print(f"Shape: {merged_df.shape}")
print("\nMissing Values:")
print(merged_df.isnull().sum())
print("\nAverage Values by Turbine:")
print(merged_df.groupby("turbine")[["wind_speed", "power", "temperature"]].mean())

## 3. Exploratory Data Analysis (`analysis.py`)
Load `data/merged_turbines_data.csv` and inspect dataset shape, sample rows (`head`/`tail`), column types (`info`), summary statistics (`describe`), missing values, and per-turbine record counts.

In [ ]:
# 1. Load merged dataset
MERGED_DATA_PATH = os.path.join(DATA_DIR, "merged_turbines_data.csv")
print(f"Loading merged dataset from: {MERGED_DATA_PATH}\n")
df = pd.read_csv(MERGED_DATA_PATH)

# 2. Dataset Shape
print("=" * 60)
print("1. DATASET SHAPE")
print("=" * 60)
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}\n")

# 3. First 5 Rows (Head)
print("=" * 60)
print("2. FIRST 5 ROWS (HEAD)")
print("=" * 60)
print(df.head())
print("\n")

# 4. Last 5 Rows (Tail)
print("=" * 60)
print("3. LAST 5 ROWS (TAIL)")
print("=" * 60)
print(df.tail())
print("\n")

# 5. Dataset Info
print("=" * 60)
print("4. DATASET INFO")
print("=" * 60)
df.info()
print("\n")

# 6. Statistical Description (Describe)
print("=" * 60)
print("5. STATISTICAL DESCRIPTION (DESCRIBE)")
print("=" * 60)
print(df.describe())
print("\n")

# 7. Missing Values
print("=" * 60)
print("6. MISSING VALUES PER COLUMN")
print("=" * 60)
print(df.isnull().sum())
print("\n")

# 8. Turbine Record Counts
print("=" * 60)
print("7. TURBINE RECORD COUNTS")
print("=" * 60)
print(df["turbine"].value_counts())
print("\n")

## 4. Data Preprocessing & Feature Engineering (`preprocess.py`)
Process all 6 turbine SCADA files into a synchronized hourly feature matrix:
- **Boundary Clipping & Imputation**: Clip power to `[0, 2050]` kW and wind speed to `[0, 50]` m/s, then interpolate missing values.
- **Wind Vector Decomposition**: Convert wind speed and direction into orthogonal horizontal vectors (`wind_u`, `wind_v`).
- **1-Hour Resampling**: Resample 10-minute telemetry to 1-hour means.
- **Theoretical Power Curve**: Compute Senvion MM92 aerodynamic power (`cut-in = 3.0 m/s`, `rated = 12.5 m/s`, `cut-out = 25.0 m/s`, `rated_power = 2050 kW`).
- **Farm Aggregates, Cyclical Time, and Lag Features**: Merge all 6 turbines, compute farm-level totals/means, add sine/cosine encodings for hour and month, and create `1h`, `2h`, and `24h` autoregressive lags.
- Save outputs to `processed_kelmarsh_hourly.parquet` and `.csv`.

In [ ]:
# Turbine MM92 Constants
RATED_POWER_KW = 2050.0
CUT_IN_SPEED_MS = 3.0
RATED_SPEED_MS = 12.5
CUT_OUT_SPEED_MS = 25.0

# 1. Load and process each turbine file
SCADA_COLS = [
    "# Date and time",
    "Wind speed (m/s)",
    "Wind direction (°)",
    "Power (kW)",
    "Nacelle ambient temperature (°C)",
]

files = sorted(glob.glob(os.path.join(DATA_DIR, "Turbine_Data_Kelmarsh_*_2024-*.csv")))
turbine_dfs = []

print("Processing all 6 turbine SCADA files...")
for i, fpath in enumerate(files, start=1):
    print(f"  Cleaning and resampling Turbine {i}...")
    df_t = pd.read_csv(fpath, skiprows=9, usecols=SCADA_COLS)
    df_t.columns = ["timestamp", "wind_speed", "wind_direction", "power", "temperature"]

    # Drop blank trailing rows
    df_t = df_t.dropna(subset=["timestamp", "wind_speed"]).copy()
    df_t["timestamp"] = pd.to_datetime(df_t["timestamp"])
    df_t = df_t.sort_values("timestamp").reset_index(drop=True)

    # Clean signal boundaries
    df_t["power"] = df_t["power"].clip(lower=0.0, upper=RATED_POWER_KW)
    df_t["wind_speed"] = df_t["wind_speed"].clip(lower=0.0, upper=50.0)

    # Impute missing values
    df_t["power"] = df_t["power"].interpolate().bfill()
    df_t["temperature"] = df_t["temperature"].interpolate().bfill()
    df_t["wind_direction"] = df_t["wind_direction"].interpolate().bfill()

    # Wind vector decomposition
    rad = np.deg2rad(df_t["wind_direction"].values)
    df_t["wind_u"] = -df_t["wind_speed"].values * np.sin(rad)
    df_t["wind_v"] = -df_t["wind_speed"].values * np.cos(rad)

    # Resample to 1-hour means
    df_t = df_t.set_index("timestamp")
    hourly = df_t.resample("1h").mean().interpolate().reset_index()

    # Calculate theoretical physical power directly
    ws = hourly["wind_speed"].values
    theo_p = np.zeros_like(ws, dtype=float)
    ramp = (ws >= CUT_IN_SPEED_MS) & (ws < RATED_SPEED_MS)
    theo_p[ramp] = RATED_POWER_KW * np.power((ws[ramp] - CUT_IN_SPEED_MS) / (RATED_SPEED_MS - CUT_IN_SPEED_MS), 3)
    rated = (ws >= RATED_SPEED_MS) & (ws <= CUT_OUT_SPEED_MS)
    theo_p[rated] = RATED_POWER_KW
    hourly["power_theoretical"] = theo_p

    # Add turbine prefix to column names
    prefix = f"t{i}_"
    hourly = hourly.rename(columns={c: f"{prefix}{c}" for c in hourly.columns if c != "timestamp"})
    turbine_dfs.append(hourly)

# 2. Merge all turbines into a single synchronized dataset
print("\nMerging all turbines into farm dataset...")
merged = turbine_dfs[0]
for t_df in turbine_dfs[1:]:
    merged = pd.merge(merged, t_df, on="timestamp", how="inner")

# 3. Add Farm Aggregates
power_cols = [f"t{i}_power" for i in range(1, 7)]
wind_cols = [f"t{i}_wind_speed" for i in range(1, 7)]
theo_cols = [f"t{i}_power_theoretical" for i in range(1, 7)]

merged["farm_total_power"] = merged[power_cols].sum(axis=1)
merged["farm_mean_wind_speed"] = merged[wind_cols].mean(axis=1)
merged["farm_theoretical_power"] = merged[theo_cols].sum(axis=1)

# 4. Cyclical Time Features
hour = merged["timestamp"].dt.hour.values
month = merged["timestamp"].dt.month.values
merged["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
merged["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
merged["month_sin"] = np.sin(2 * np.pi * month / 12.0)
merged["month_cos"] = np.cos(2 * np.pi * month / 12.0)

# 5. Temporal Lag Features
for lag in [1, 2, 24]:
    merged[f"farm_power_lag_{lag}h"] = merged["farm_total_power"].shift(lag)
    merged[f"farm_wind_lag_{lag}h"] = merged["farm_mean_wind_speed"].shift(lag)

merged = merged.dropna().reset_index(drop=True)

# 6. Save Processed Datasets
parquet_path = os.path.join(DATA_DIR, "processed_kelmarsh_hourly.parquet")
csv_path = os.path.join(DATA_DIR, "processed_kelmarsh_hourly.csv")
merged.to_parquet(parquet_path, index=False)
merged.to_csv(csv_path, index=False)
print(f"Saved processed dataset: {len(merged):,} rows, {merged.shape[1]} columns")
print("SCADA preprocessing complete.")

## 5. Data Visualization
Visualize inter-turbine generation distributions (`plot_turbine_comparison.py`) and diurnal/seasonal farm generation patterns (`plot_diurnal_seasonal_heatmap.py`).

### 5.1 Inter-Turbine Generation Comparison
Compare hourly power distributions (boxplot) and mean power output (bar chart) across all 6 turbines.

In [ ]:
PROCESSED_PARQUET = os.path.join(DATA_DIR, "processed_kelmarsh_hourly.parquet")
PROCESSED_CSV = os.path.join(DATA_DIR, "processed_kelmarsh_hourly.csv")
TURBINE_PLOT_PATH = os.path.join(DATA_DIR, "turbine_comparison.png")

if os.path.exists(PROCESSED_PARQUET):
    df_vis = pd.read_parquet(PROCESSED_PARQUET)
else:
    df_vis = pd.read_csv(PROCESSED_CSV)

turbines = [f"t{i}" for i in range(1, 7)]
labels = [f"Turbine {i}" for i in range(1, 7)]
vis_power_cols = [f"{t}_power" for t in turbines]

# Convert power values to kW and compute summary metrics
power_data = [df_vis[col].values for col in vis_power_cols]
mean_powers_kw = [df_vis[col].mean() for col in vis_power_cols]

# Create figure with 2 subplots (Boxplot + Mean Generation Bar Chart)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Distribution boxplot
box = ax1.boxplot(
    power_data,
    tick_labels=labels,
    patch_artist=True,
    showmeans=True,
    meanprops={"marker": "o", "markerfacecolor": "red", "markeredgecolor": "red"}
)
for patch in box["boxes"]:
    patch.set_facecolor("#6baed6")

ax1.set_title("Hourly Power Distribution per Turbine", fontsize=12)
ax1.set_ylabel("Power Output (kW)", fontsize=11)
ax1.grid(axis="y", linestyle="--", alpha=0.6)

# Subplot 2: Mean output comparison
bars = ax2.bar(labels, mean_powers_kw, color="#3182bd", edgecolor="black", alpha=0.85)
ax2.set_title("Mean Power Output per Turbine", fontsize=12)
ax2.set_ylabel("Mean Power (kW)", fontsize=11)
ax2.grid(axis="y", linestyle="--", alpha=0.6)

for bar in bars:
    height = bar.get_height()
    ax2.annotate(
        f"{height:.1f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.suptitle("Kelmarsh Wind Farm: Inter-Turbine Generation Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(TURBINE_PLOT_PATH, dpi=300, bbox_inches="tight")
plt.show()
print(f"Turbine comparison plot saved to: {TURBINE_PLOT_PATH}")

### 5.2 Diurnal & Seasonal Generation Heatmap
Plot mean farm generation (in MW) grouped by month and hour of day (UTC).

In [ ]:
HEATMAP_PATH = os.path.join(DATA_DIR, "diurnal_seasonal_heatmap.png")

df_vis["timestamp"] = pd.to_datetime(df_vis["timestamp"])
df_vis["hour"] = df_vis["timestamp"].dt.hour
df_vis["month"] = df_vis["timestamp"].dt.month

# Compute mean power (kW) by month and hour
pivot = df_vis.pivot_table(
    index="month",
    columns="hour",
    values="farm_total_power",
    aggfunc="mean"
)

# Convert to MW for readability
pivot_mw = pivot / 1000.0

# Plotting heatmap
fig, ax = plt.subplots(figsize=(12, 6))
cax = ax.imshow(pivot_mw, cmap="viridis", aspect="auto", origin="lower")

cbar = fig.colorbar(cax, ax=ax)
cbar.set_label("Mean Farm Generation (MW)", fontsize=11)

month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ax.set_yticks(range(len(pivot_mw.index)))
ax.set_yticklabels([month_labels[m - 1] for m in pivot_mw.index])

ax.set_xticks(range(24))
ax.set_xlabel("Hour of Day (UTC)", fontsize=11)
ax.set_ylabel("Month", fontsize=11)
ax.set_title("Kelmarsh Wind Farm: Diurnal and Seasonal Generation Heatmap", fontsize=13, pad=12)

plt.tight_layout()
plt.savefig(HEATMAP_PATH, dpi=300)
plt.show()
print(f"Heatmap saved to: {HEATMAP_PATH}")

## 6. Chronological Splitting & Feature Scaling (`prepare_scaling.py`)
Split the processed hourly dataset chronologically into **70% Train / 15% Validation / 15% Test** to prevent temporal data leakage. Fit a `StandardScaler` strictly on the training split, transform all splits, and save both the scaler artifact and scaled Parquet files.

In [ ]:
# 1. Load preprocessed data
if os.path.exists(PROCESSED_PARQUET):
    df_prep = pd.read_parquet(PROCESSED_PARQUET)
else:
    df_prep = pd.read_csv(PROCESSED_CSV)

df_prep["timestamp"] = pd.to_datetime(df_prep["timestamp"])
df_prep = df_prep.sort_values("timestamp").reset_index(drop=True)

# 2. Chronological 70% Train, 15% Val, 15% Test split (No Data Leakage)
n = len(df_prep)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_prep.iloc[:train_end].copy()
val_df = df_prep.iloc[train_end:val_end].copy()
test_df = df_prep.iloc[val_end:].copy()

print(f"Data Split -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# 3. Identify feature columns (exclude timestamp)
feature_cols = [c for c in df_prep.columns if c != "timestamp"]

# 4. Fit StandardScaler strictly on training set
scaler = StandardScaler()
scaler.fit(train_df[feature_cols])

# 5. Transform all splits
train_scaled = train_df.copy()
val_scaled = val_df.copy()
test_scaled = test_df.copy()

train_scaled[feature_cols] = scaler.transform(train_df[feature_cols])
val_scaled[feature_cols] = scaler.transform(val_df[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

# 6. Save scaler artifact and scaled datasets
scaler_path = os.path.join(MODELS_DIR, "feature_scaler.joblib")
joblib.dump({"scaler": scaler, "feature_cols": feature_cols}, scaler_path)

train_scaled.to_parquet(os.path.join(DATA_DIR, "train_scaled.parquet"), index=False)
val_scaled.to_parquet(os.path.join(DATA_DIR, "val_scaled.parquet"), index=False)
test_scaled.to_parquet(os.path.join(DATA_DIR, "test_scaled.parquet"), index=False)

print(f"StandardScaler saved to: {scaler_path}")
print("Saved train_scaled, val_scaled, and test_scaled files to data directory.")

## 7. Baseline MLP Model Training & Evaluation (`baseline_model.py`)
Load the scaled splits and `feature_scaler.joblib`, train a 3-hidden-layer PyTorch Multi-Layer Perceptron (`128 -> 64 -> 32 -> 1` with ReLU and Dropout `0.2`) for 40 epochs using Adam (`lr=0.001`) and MSE loss, checkpoint the best validation model, and evaluate on the held-out test split in both Z-score and physical kW units.

In [ ]:
# 1. Load Scaled Data & Scaler Artifact
train_df = pd.read_parquet(os.path.join(DATA_DIR, "train_scaled.parquet"))
val_df = pd.read_parquet(os.path.join(DATA_DIR, "val_scaled.parquet"))
test_df = pd.read_parquet(os.path.join(DATA_DIR, "test_scaled.parquet"))

scaler_meta = joblib.load(os.path.join(MODELS_DIR, "feature_scaler.joblib"))
scaler = scaler_meta["scaler"]
feature_cols = scaler_meta["feature_cols"]

target_col = "farm_total_power"
target_idx = feature_cols.index(target_col)
target_mean = scaler.mean_[target_idx]
target_scale = scaler.scale_[target_idx]

input_features = [c for c in feature_cols if c != target_col]

X_train = train_df[input_features].values.astype(np.float32)
y_train = train_df[target_col].values.astype(np.float32)

X_val = val_df[input_features].values.astype(np.float32)
y_val = val_df[target_col].values.astype(np.float32)

X_test = test_df[input_features].values.astype(np.float32)
y_test = test_df[target_col].values.astype(np.float32)

# Unscaled true test power for realistic kW evaluation
y_test_kw = (y_test * target_scale) + target_mean

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=64, shuffle=False)

# 2. Define Baseline Multi-Layer Perceptron (MLP)
class BaselineMLP(nn.Module):
    def __init__(self, in_features):
        super(BaselineMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BaselineMLP(len(input_features)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 3. Train Baseline Model
print("Training Baseline MLP...")
epochs = 40
best_val_loss = float("inf")
best_model_path = os.path.join(MODELS_DIR, "baseline_mlp.pth")

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        pred = model(bx)
        loss = criterion(pred, by)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(bx)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            val_loss += criterion(model(bx), by).item() * len(bx)
    val_loss /= len(val_loader.dataset)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

In [ ]:
# 4. Evaluate on Unseen Test Split
model.load_state_dict(torch.load(best_model_path, weights_only=True))
model.eval()
with torch.no_grad():
    test_pred_scaled = model(torch.tensor(X_test).to(device)).cpu().numpy()

# Invert target scaling back to physical kW
test_pred_kw = (test_pred_scaled * target_scale) + target_mean
test_pred_kw = np.clip(test_pred_kw, 0.0, 12300.0)

# Compute errors in Z-score units (model raw space)
mae_z = mean_absolute_error(y_test, test_pred_scaled)
rmse_z = root_mean_squared_error(y_test, test_pred_scaled)

mae = mean_absolute_error(y_test_kw, test_pred_kw)
rmse = root_mean_squared_error(y_test_kw, test_pred_kw)
r2 = r2_score(y_test_kw, test_pred_kw)
farm_capacity_kw = 12300.0
normalized_mae = (mae / farm_capacity_kw) * 100.0

# Compute Accuracy metrics
# 1. Capacity-based accuracy (100% - Normalized MAE)
accuracy_capacity = 100.0 - normalized_mae

# 2. Mean Absolute Percentage Accuracy on non-idle generation (>100 kW)
mask_active = y_test_kw > 100.0
mape_active = np.mean(np.abs((y_test_kw[mask_active] - test_pred_kw[mask_active]) / y_test_kw[mask_active])) * 100.0
accuracy_mape = max(0.0, 100.0 - mape_active)

print("\n--- BASELINE MLP TEST PERFORMANCE ---")
print("1. Raw Scaled (Z-Score) Space:")
print(f"   MAE (Z-Score):  {mae_z:.4f}")
print(f"   RMSE (Z-Score): {rmse_z:.4f}")

print("\n2. Physical Electricity (Kilowatts) Space:")
print(f"   MAE:            {mae:.2f} kW ({normalized_mae:.2f}% of rated capacity)")
print(f"   RMSE:           {rmse:.2f} kW")
print(f"   Accuracy (Cap): {accuracy_capacity:.2f}%")
print(f"   Accuracy (Act): {accuracy_mape:.2f}%")
print(f"   R^2 Score:      {r2:.4f}")

# Sample table showing actual Z-score vs actual kW comparisons
print("\n3. Sample Predictions (First 5 Test Hours):")
print(f"{'Hour':<6}{'Actual Z':<12}{'Pred Z':<12}{'Z Error':<12}{'Actual kW':<14}{'Pred kW':<14}{'kW Error':<10}")
print("-" * 80)
for i in range(5):
    act_z = y_test[i]
    pred_z = test_pred_scaled[i]
    err_z = abs(act_z - pred_z)
    act_kw = y_test_kw[i]
    pred_kw = test_pred_kw[i]
    err_kw = abs(act_kw - pred_kw)
    print(f"{i+1:<6}{act_z:<12.4f}{pred_z:<12.4f}{err_z:<12.4f}{act_kw:<14.2f}{pred_kw:<14.2f}{err_kw:<10.2f}")

print(f"\nModel saved to: {best_model_path}")

## 8. Baseline Random Forest Model & Comparison with MLP
Train a `RandomForestRegressor` baseline on the same scaled training split (`X_train`, `y_train`), evaluate it on `X_test` using the same Z-score and physical kW metrics, and compare its accuracy against the Baseline MLP.

In [ ]:
# 1. Train Baseline Random Forest Model
print("Training Baseline Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_model_path = os.path.join(MODELS_DIR, "baseline_rf.joblib")
joblib.dump(rf_model, rf_model_path)

# 2. Evaluate on Unseen Test Split
rf_pred_scaled = rf_model.predict(X_test)
rf_pred_kw = (rf_pred_scaled * target_scale) + target_mean
rf_pred_kw = np.clip(rf_pred_kw, 0.0, 12300.0)

# Compute Z-score metrics
rf_mae_z = mean_absolute_error(y_test, rf_pred_scaled)
rf_rmse_z = root_mean_squared_error(y_test, rf_pred_scaled)

# Compute physical kW metrics
rf_mae = mean_absolute_error(y_test_kw, rf_pred_kw)
rf_rmse = root_mean_squared_error(y_test_kw, rf_pred_kw)
rf_r2 = r2_score(y_test_kw, rf_pred_kw)
rf_normalized_mae = (rf_mae / farm_capacity_kw) * 100.0

rf_accuracy_capacity = 100.0 - rf_normalized_mae
rf_mape_active = np.mean(np.abs((y_test_kw[mask_active] - rf_pred_kw[mask_active]) / y_test_kw[mask_active])) * 100.0
rf_accuracy_mape = max(0.0, 100.0 - rf_mape_active)

print("\n--- BASELINE RANDOM FOREST TEST PERFORMANCE ---")
print("1. Raw Scaled (Z-Score) Space:")
print(f"   MAE (Z-Score):  {rf_mae_z:.4f}")
print(f"   RMSE (Z-Score): {rf_rmse_z:.4f}")

print("\n2. Physical Electricity (Kilowatts) Space:")
print(f"   MAE:            {rf_mae:.2f} kW ({rf_normalized_mae:.2f}% of rated capacity)")
print(f"   RMSE:           {rf_rmse:.2f} kW")
print(f"   Accuracy (Cap): {rf_accuracy_capacity:.2f}%")
print(f"   Accuracy (Act): {rf_accuracy_mape:.2f}%")
print(f"   R^2 Score:      {rf_r2:.4f}")
print(f"\nRandom Forest model saved to: {rf_model_path}")

### 8.1 Model Comparison: Baseline MLP vs. Random Forest
Compare test-set accuracy, MAE, RMSE, and $R^2$ between the Baseline MLP and Random Forest models.

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Baseline MLP",
        "MAE (Z)": round(mae_z, 4),
        "RMSE (Z)": round(rmse_z, 4),
        "MAE (kW)": round(mae, 2),
        "RMSE (kW)": round(rmse, 2),
        "Norm MAE (%)": round(normalized_mae, 2),
        "Accuracy Cap (%)": round(accuracy_capacity, 2),
        "Accuracy Act (%)": round(accuracy_mape, 2),
        "R2": round(r2, 4),
    },
    {
        "Model": "Random Forest",
        "MAE (Z)": round(rf_mae_z, 4),
        "RMSE (Z)": round(rf_rmse_z, 4),
        "MAE (kW)": round(rf_mae, 2),
        "RMSE (kW)": round(rf_rmse, 2),
        "Norm MAE (%)": round(rf_normalized_mae, 2),
        "Accuracy Cap (%)": round(rf_accuracy_capacity, 2),
        "Accuracy Act (%)": round(rf_accuracy_mape, 2),
        "R2": round(rf_r2, 4),
    },
])

print("--- MODEL COMPARISON (TEST SET) ---")
print(comparison_df.to_string(index=False))

better_model = "Random Forest" if rf_mae < mae else "Baseline MLP"
print(f"\nLower Test MAE: {better_model} ({min(mae, rf_mae):.2f} kW vs {max(mae, rf_mae):.2f} kW)")

# Visual comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
models = comparison_df["Model"]
x = np.arange(len(models))
width = 0.35

# Errors (kW)
ax1.bar(x - width / 2, comparison_df["MAE (kW)"], width, label="MAE (kW)", color="#3182bd")
ax1.bar(x + width / 2, comparison_df["RMSE (kW)"], width, label="RMSE (kW)", color="#9ecae1")
ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.set_ylabel("Error (kW)")
ax1.set_title("Test Error Comparison (Lower is Better)")
ax1.legend()
ax1.grid(axis="y", linestyle="--", alpha=0.5)

# Accuracies (%)
ax2.bar(x - width / 2, comparison_df["Accuracy Cap (%)"], width, label="Accuracy (Cap %)", color="#31a354")
ax2.bar(x + width / 2, comparison_df["Accuracy Act (%)"], width, label="Accuracy (Active %)", color="#a1d99b")
ax2.set_xticks(x)
ax2.set_xticklabels(models)
ax2.set_ylabel("Accuracy (%)")
ax2.set_ylim(0, 105)
ax2.set_title("Test Accuracy Comparison (Higher is Better)")
ax2.legend()
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()